## About this notebook — `scanned_small_pipeline.ipynb`

**Purpose:** Compact, fast interactive pipeline for high-quality **scanned** fingerprint images.
Uses sensible defaults — intended for students getting started or for quick parameter exploration.

| | |
|---|---|
| **Input** | High-quality scanned fingerprint (white background, high contrast) |
| **Output** | Skeleton overlay, minutiae visualisation, saved minutiae JSON |
| **Pipeline** | ipywidgets sliders — re-executes on every slider change |
| **Best for** | Quick first look at a scanned fingerprint; learning the basics |

### Methods used
`variance ROI mask` · `CLAHE` · `adaptive binarization` · `morphology cleanup` ·
`skeletonization` · `minutiae extraction (crossing number)`

---

### Notebook comparison

| | Notebook | Input | Output | Pipeline | Best for |
|---|:---|:---|:---|:---|:---|
| | scanning | Live scanner | Captured images | Buttons | Acquire fingerprints from hardware |
| | photo_small_pipeline | Photo (dark background) | Ridge/valley map + skeleton | Widgets + polygon ROI | Photo processing with scale calibration |
| ▶ | **scanned_small_pipeline** | Scanned image | Skeleton + minutiae | Widget sliders | Quick processing, sensible defaults |
| | scanned_full_pipeline | Scanned image | Skeleton + minutiae | Widget sliders + toggles | Full parameter control, scanned FP |
| | minutiae_comparison | Minutiae JSON files | Comparison image + score | Alignment + matching | Compare real vs fake fingerprints |
| | algorithms_explorer | Any | Side-by-side comparisons | Static comparisons | Learn and compare all available algorithms |


In [ ]:
import sys, importlib
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import fingerprints.pipeline_presets
import fingerprints.interactive
importlib.reload(fingerprints.interactive)
importlib.reload(fingerprints.pipeline_presets)

import fingerprints.pipeline_presets as pp
import fingerprints.interactive as ip

print('PROJECT_ROOT:', PROJECT_ROOT)
print('interactive: ', ip.__file__)


In [ ]:
# ── CONFIGURE HERE ────────────────────────────────────────────────────────────
UCO       = '232886'  # your UCO number
N         = 1          # scan version (1, 2, 3 ...)
FINGER    = ''         # finger name, e.g. 'index', 'thumb' — leave empty to list available
SCAN_TYPE = 'real'     # 'real'  → data/scans/real/   (real finger scan)
                       # 'fake'  → data/scans/fake/   (silicone / printed fake)
# ──────────────────────────────────────────────────────────────────────────────


In [ ]:
import re
import cv2
import matplotlib.pyplot as plt

_errors = []
if not str(UCO).isdigit():
    _errors.append(f'  UCO "{UCO}" must be digits only')
if not isinstance(N, int) or N < 1:
    _errors.append(f'  N "{N}" must be a positive integer')

if _errors:
    print('Config errors — fix the config cell above and re-run:')
    for e in _errors: print(e)
    IMAGE_PATH = None; img = None
else:
    SCANS_DIR = PROJECT_ROOT / 'data' / 'scans' / SCAN_TYPE

    # list all files matching any finger for this UCO/N
    _pat_any = re.compile(rf'^[a-zA-Z]+_{re.escape(str(UCO))}_{N}\.(jpeg|jpg|png|bmp|tiff?)$', re.IGNORECASE)
    _all = sorted(p for p in SCANS_DIR.iterdir() if _pat_any.match(p.name))

    if not _all:
        print(f'No files found in {SCANS_DIR} for UCO={UCO}, N={N}')
        print(f'Expected e.g.: index_{UCO}_{N}.jpeg')
        IMAGE_PATH = None; img = None
    elif not FINGER:
        print(f'Available fingerprints for UCO={UCO}, N={N}:')
        for p in _all: print(f'  {p.name}')
        print('Set FINGER in the config cell to one of the names above, then re-run.')
        IMAGE_PATH = None; img = None
    else:
        _pat_finger = re.compile(rf'^{re.escape(FINGER.lower())}_{re.escape(str(UCO))}_{N}\.(jpeg|jpg|png|bmp|tiff?)$', re.IGNORECASE)
        _match = next((p for p in _all if _pat_finger.match(p.name.lower())), None)
        if _match is None:
            print(f'Finger "{FINGER}" not found for UCO={UCO}, N={N}.')
            print(f'Available: {[p.name for p in _all]}')
            IMAGE_PATH = None; img = None
        else:
            IMAGE_PATH = _match
            img = cv2.imread(str(IMAGE_PATH), cv2.IMREAD_GRAYSCALE)
            if img is None:
                raise RuntimeError(f'Failed to load: {IMAGE_PATH}')
            print(f'Loaded: {IMAGE_PATH.name}  shape={img.shape}  dtype={img.dtype}')

            fig, ax = plt.subplots(figsize=(5, 7))
            ax.imshow(img, cmap='gray')
            ax.set_title(IMAGE_PATH.name)
            ax.axis('off')
            plt.tight_layout()
            plt.show()


In [ ]:
if img is None:
    print('No image loaded — fix the config cell and re-run.')
else:
    STEPS = [
        'roi_segmentation_vis',   # variance-based ROI mask
        'apply_mask_zero',        # zero out background
        'clahe',                  # contrast enhancement
        'binarize',               # ridge binarization
        'mask_binary',            # re-apply mask to binary
        'morphology',             # optional noise cleanup (all zeros = no change)
        'skeleton_vis',           # ridge thinning + overlay
        'prune_skeleton',         # optional spur removal
        'minutiae_vis',           # minutiae detection + visualization
    ]

    ENABLED = {
        'roi_segmentation_vis': True,
        'apply_mask_zero':      True,
        'clahe':                True,
        'binarize':             True,
        'mask_binary':          True,
        'morphology':           True,
        'skeleton_vis':         True,
        'prune_skeleton':       False,
        'minutiae_vis':         True,
    }

    OVERRIDES = {
        'roi_segmentation_vis': {
            'block':      {'kind': 'int',   'value': 16,   'min': 8,   'max': 64,    'step': 4,   'label': 'Block'},
            'close_k':    {'kind': 'int',   'value': 25,   'min': 1,   'max': 51,    'step': 2,   'label': 'Close K'},
            'erode_k':    {'kind': 'int',   'value': 7,    'min': 1,   'max': 21,    'step': 2,   'label': 'Erode K'},
            'min_area':   {'kind': 'int',   'value': 4000, 'min': 100, 'max': 20000, 'step': 100, 'label': 'Min Area'},
        },
        'clahe': {
            'clip': {'kind': 'float', 'value': 2.0, 'min': 0.5, 'max': 8.0, 'step': 0.1, 'label': 'Clip'},
            'tile': {'kind': 'int',   'value': 8,   'min': 2,   'max': 32,  'step': 2,   'label': 'Tile'},
        },
        'binarize': {
            'method':     {'kind': 'choice', 'value': 'adaptive',
                           'options': ['adaptive', 'otsu', 'sauvola', 'niblack', 'fixed'],
                           'label': 'Method'},
            'block':      {'kind': 'int',   'value': 31,  'min': 9,  'max': 61,  'step': 2, 'label': 'Block'},
            'C':          {'kind': 'int',   'value': 5,   'min': -5, 'max': 15,  'step': 1, 'label': 'C'},
            'blur_ksize': {'kind': 'int',   'value': 3,   'min': 1,  'max': 9,   'step': 2, 'label': 'Blur K'},
        },
        'morphology': {
            'open_k':   {'kind': 'int',  'value': 0, 'min': 0, 'max': 7, 'step': 1, 'label': 'Open K'},
            'close_k':  {'kind': 'int',  'value': 0, 'min': 0, 'max': 7, 'step': 1, 'label': 'Close K'},
            'min_area': {'kind': 'int',  'value': 0, 'min': 0, 'max': 1000, 'step': 10, 'label': 'Min Area'},
            'fill':     {'kind': 'bool', 'value': False, 'label': 'Fill Holes'},
        },
        'minutiae_vis': {
            'border_margin':  {'kind': 'int',  'value': 10, 'min': 0,  'max': 50, 'step': 1, 'label': 'Border Margin'},
            'cluster_dist':   {'kind': 'int',  'value': 6,  'min': 0,  'max': 20, 'step': 1, 'label': 'Cluster Dist'},
            'max_spur_len':   {'kind': 'int',  'value': 15, 'min': 0,  'max': 30, 'step': 1, 'label': 'Max Spur Len'},
            'suppress_dist':  {'kind': 'int',  'value': 12, 'min': 0,  'max': 20, 'step': 1, 'label': 'Suppress Dist'},
        },
    }

    pipeline = pp.make_custom_pipeline(
        img,
        order=STEPS,
        enabled_map=ENABLED,
        per_step_param_overrides=OVERRIDES,
        preset_name='scan_small',
    )
    display(pipeline.display())


## Save results

Run this cell after tuning the pipeline to save the skeleton overlay and minutiae to disk.


In [ ]:
import json as _json
import numpy as np
from PIL import Image as _PilImage

if IMAGE_PATH is None:
    print('No image loaded.')
elif pipeline is None:
    print('Pipeline not built.')
else:
    stem = IMAGE_PATH.stem          # e.g. 'index_232886_1'
    OUT_DIR = IMAGE_PATH.parent / 'processed'
    OUT_DIR.mkdir(parents=True, exist_ok=True)

    # ── skeleton overlay ──────────────────────────────────────────────────────
    skel_vis = pipeline.context.get('skeleton_vis_img')
    if skel_vis is not None:
        skel_path = OUT_DIR / f'{stem}_skeleton.png'
        _PilImage.fromarray(skel_vis if skel_vis.dtype == 'uint8'
                            else (skel_vis * 255).astype('uint8')).save(str(skel_path))
        print(f'Skeleton → {skel_path}')
    else:
        print('skeleton_vis_img not found — run the pipeline first.')

    # ── minutiae JSON ─────────────────────────────────────────────────────────
    minutiae = pipeline.context.get('minutiae_list')
    if minutiae is not None:
        def _serialisable(m):
            return {k: (float(v) if isinstance(v, (float, np.floating)) else
                        int(v)   if isinstance(v, (int,   np.integer))  else v)
                    for k, v in m.items()}

        out = {
            'image':    IMAGE_PATH.name,
            'uco':      UCO,
            'n':        N,
            'finger':   FINGER,
            'count':    len(minutiae),
            'minutiae': [_serialisable(m) for m in minutiae],
        }
        min_path = OUT_DIR / f'{stem}_minutiae.json'
        with open(min_path, 'w') as _f:
            _json.dump(out, _f, indent=2)
        print(f'Minutiae  → {min_path}  ({len(minutiae)} points)')
    else:
        print('minutiae_list not found — enable the Minutiae Extraction step and re-run.')


## Save / reload pipeline parameters

After tuning the sliders, save the current settings to JSON so you can restore them later.


In [ ]:
PARAMS_PATH = PROJECT_ROOT / f'scan_small_{UCO}_{N}_{FINGER}_params.json'
pipeline.export_params_json(str(PARAMS_PATH))
print('Saved:', PARAMS_PATH)


In [ ]:
PARAMS_PATH = PROJECT_ROOT / f'scan_small_{UCO}_{N}_{FINGER}_params.json'
if PARAMS_PATH.exists():
    pipeline.import_params_json(str(PARAMS_PATH), rerun=True)
    print('Loaded:', PARAMS_PATH)
else:
    print('Not found yet:', PARAMS_PATH)
